In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (HEPAD)
This notebook curates the **HEPAD** dataset by integrating peptide sequences distributed across multiple subfolders (Hmp1, Hmp2, Hmp3, and Hmpm). Sequences are parsed from FASTA files, binary hemolysis labels are inferred from filename conventions, duplicate consistency checks are applied, and the final curated dataset and metadata are exported for downstream analysis.

- **Toxic effect / endpoint:** hemolytic
- **Source:** HEPAD
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:- **Loads FASTA files** (`.fasta`) from the HEPAD subfolders:
  - `Hmp1/`, `Hmp2/`, `Hmp3/`, and `Hmpm/`.
- **Parses peptide sequences** using a unified FASTA reader and tracks the originating filename.
- **Infers hemolytic labels from filenames**:
  - filenames containing `"neg"` (case-insensitive) → `label = 0`,
  - all other files are treated as hemolytic (`label = 1`).
- **Keeps a standardized schema**:
  - `sequence`
  - `label`
- **Checks duplicated sequences** across all inputs:
  - unique sequences are retained,
  - duplicates with consistent labels are collapsed,
  - sequences with conflicting labels are flagged as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated outputs**:
  - `processed_hemolytic_dataset.csv`,
  - `detected_error_sequences.csv`,
  - `metadata.json`.

In [2]:
name_source = "HEPAD"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants.
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
folders = ["Hmp1", "Hmp2", "Hmp3", "Hmpm"]
dfs = []

for folder in folders:
    for file in (Path(PATH_INPUT) / name_source / folder).glob("*.fasta"):
        df = read_fasta_doc(file)
        df["source_file"] = file.name
        dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

In [4]:
df = (
    df
    .assign(
        label=lambda d: d["source_file"]
            .str.contains("neg", case=False, na=False)
            .map({True: 0, False: 1})
    )
    [["sequence","label"]]
)
df.shape

(5570, 2)

- Checking duplicates

In [5]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [6]:
df_full.shape

(2996, 2)

In [7]:
df_errors.shape

(84, 1)

- Working with metada

In [8]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [9]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences" : len(df_errors),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2025,
 'last update date': datetime.datetime(2024, 4, 21, 0, 0),
 'download date': Timestamp('2025-10-17 00:00:00'),
 'file format': 'fasta',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'Negative;Positive',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from Swiss-Prot',
 'repository or server': 'https://github.com/csh07/HEPAD/tree/main/data',
 'publication': 'https://link.springer.com/article/10.1186/s12859-025-06254-6',
 'number_of_raw_sequences': 5570,
 'number_of_sequences_retained': 2996,
 'number_of_positive_sequences': 910,
 'number_of_negative_sequences': 2086,
 'number_of_erroneous_sequences': 84,
 'modified_sequences_included': False}

- Exporting data

In [10]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [11]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)